In [1]:
import sys
from pathlib import Path

# Add the src directory to Python path
project_root = Path().resolve().parent
src_path = project_root / "src"
sys.path.insert(0, str(src_path))


In [2]:
# Standard library imports
import re
import sys
from datetime import datetime
from pathlib import Path

# Third-party imports
import requests
from bs4 import BeautifulSoup

# Local imports
from common.spark_session import get_spark


In [3]:
# Configuration
project_root = Path().resolve().parent
data_dir = project_root / "data" / "raw"
data_dir.mkdir(parents=True, exist_ok=True)

# Years to check
current_year = datetime.now().year
years = list(range(2019, current_year + 1))
available_years = []


In [4]:
# Discover and download Parquet files
base_url = "https://s3.sa-east-1.amazonaws.com/ckan.saude.gov.br/SRAG"

# Get dates from OpenDataSUS website (banco congelado and banco ativo)
freeze_date_str = None
live_date_str = None
page_url = "https://opendatasus.saude.gov.br/dataset/srag-2021-a-2024"

try:
    response = requests.get(page_url, timeout=10)
    response.raise_for_status()
    soup = BeautifulSoup(response.content, "html.parser")

    # Find all dates using date structure DD/MM/YYYY
    date_pattern = r"\d{2}/\d{2}/\d{4}"
    all_dates = set()

    for text in soup.stripped_strings:
        dates = re.findall(date_pattern, text)
        all_dates.update(dates)

    # Match dates to correct entries (case-insensitive)
    for text in soup.stripped_strings:
        text_lower = text.lower()
        date_match = re.search(date_pattern, text)

        if date_match:
            date_str = date_match.group(0)  # e.g., "26/06/2025" or "10/11/2025"
            date_formatted = date_str.replace("/", "-")  # Convert to DD-MM-YYYY

            # banco congelado uses Parquet files
            is_congelado = (
                "congelado" in text_lower
                and "parquet" in text_lower
                and not freeze_date_str
            )
            if is_congelado:
                freeze_date_str = date_formatted
                print(f"Found freeze date (banco congelado): {freeze_date_str}")

            # banco vivo/ativo uses CSV files
            is_vivo = (
                ("vivo" in text_lower or "ativo" in text_lower)
                and "csv" in text_lower
                and not live_date_str
            )
            if is_vivo:
                live_date_str = date_formatted
                print(f"Found live date (banco vivo/ativo): {live_date_str}")

            if freeze_date_str and live_date_str:
                break
except Exception as e:
    print(f"Error getting dates from website: {e}")

if not freeze_date_str:
    print("Could not determine freeze date from website")
if not live_date_str:
    print("Could not determine live date from website")

# Download files for all years using appropriate date and file type
for year in years:
    year_str = str(year)[2:]
    year_dir = data_dir / str(year)
    year_dir.mkdir(exist_ok=True)

    # Determine file extension: CSV for current year (vivo), Parquet for others
    file_ext = "csv" if year == current_year else "parquet"

    # Check if file already exists locally
    existing_files = list(year_dir.glob(f"*.{file_ext}"))
    if existing_files:
        print(f"Found local file for {year}: {existing_files[0].name}")
        available_years.append(year)
        continue

    # Use live date only for current year, freeze date for all other years
    date_str = live_date_str if year == current_year else freeze_date_str

    if date_str:
        filename = f"INFLUD{year_str}-{date_str}.{file_ext}"
        url = f"{base_url}/{year}/{filename}"
        try:
            response = requests.head(url, timeout=10)
            if response.status_code == 200:
                print(f"Downloading {year}: {filename}...")
                file_response = requests.get(url, timeout=300)
                file_response.raise_for_status()
                file_path = year_dir / filename
                file_path.write_bytes(file_response.content)
                print(f"Downloaded: {file_path}")
                available_years.append(year)
            else:
                print(f"Could not find {file_ext.upper()} file for year {year}")
        except Exception as e:
            print(f"Error downloading {year}: {e}")
    else:
        print(f"Could not find file for year {year} (date unknown)")

print(f"\nAvailable years: {available_years}")


Found freeze date (banco congelado): 26-06-2025
Found live date (banco vivo/ativo): 10-11-2025
Found local file for 2019: INFLUD19-26-06-2025.parquet
Found local file for 2020: INFLUD20-26-06-2025.parquet
Found local file for 2021: INFLUD21-26-06-2025.parquet
Found local file for 2022: INFLUD22-26-06-2025.parquet
Found local file for 2023: INFLUD23-26-06-2025.parquet
Found local file for 2024: INFLUD24-26-06-2025.parquet
Downloaded: /Users/indicium.tech/Desktop/Trabalho/AI Engineering/Desafio/data/raw/2025/INFLUD25-10-11-2025.csv

Available years: [2019, 2020, 2021, 2022, 2023, 2024, 2025]


In [5]:
# Initialize Spark session
spark = get_spark(app_name="SRAG-EDA")


Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
25/11/13 14:41:55 WARN Utils: Your hostname, MacBook-Pro-de-Joao-3.local, resolves to a loopback address: 127.0.0.1; using 10.111.201.226 instead (on interface en0)
25/11/13 14:41:55 WARN Utils: Set SPARK_LOCAL_IP if you need to bind to another address
Using Spark's default log4j profile: org/apache/spark/log4j2-defaults.properties
Setting default log level to "WARN".
To adjust logging level use sc.setLogLevel(newLevel). For SparkR, use setLogLevel(newLevel).
25/11/13 14:42:00 WARN NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable


In [8]:
# Read all files and join them (CSV for current year, Parquet for others)
dataframes = []

for year in available_years:
    year_dir = data_dir / str(year)
    file_ext = "csv" if year == current_year else "parquet"
    files = list(year_dir.glob(f"*.{file_ext}"))

    if files:
        file_path = files[0]  # Take first file found
        print(f"Reading {year}: {file_path.name}")
        if file_ext == "csv":
            df = spark.read.csv(
                str(file_path),
                header=True,
                inferSchema=True,
                sep=";",  # Brazilian CSV files use semicolon delimiter
                quote='"',
                escape='"',
            )
        else:
            df = spark.read.parquet(str(file_path))
        dataframes.append(df)

# Join all dataframes
if dataframes:
    print(f"\nJoining {len(dataframes)} dataframes...")
    combined_df = dataframes[0]
    for df in dataframes[1:]:
        combined_df = combined_df.unionByName(df, allowMissingColumns=True)
    print("Join completed!")
else:
    print("No dataframes to join!")


Reading 2019: INFLUD19-26-06-2025.parquet
Reading 2020: INFLUD20-26-06-2025.parquet
Reading 2021: INFLUD21-26-06-2025.parquet
Reading 2022: INFLUD22-26-06-2025.parquet
Reading 2023: INFLUD23-26-06-2025.parquet
Reading 2024: INFLUD24-26-06-2025.parquet
Reading 2025: INFLUD25-10-11-2025.csv



Joining 7 dataframes...
Join completed!


In [9]:
# Display head
if 'combined_df' in locals():
    print(f"Total rows: {combined_df.count()}")
    print(f"Total columns: {len(combined_df.columns)}")
    print("\nFirst 20 rows:")
    combined_df.show(20, truncate=False)


Total rows: 4404275
Total columns: 194

First 20 rows:
+------------+----------+-------+----------+-------+---------+------------------------------+----------+---------------------+----------+-------+----------+----------+--------+---------+----------+-------+---------+----------+-------+-------+-----+------------------------------+----------+-----------------------+----------+-------+----------+---------+-----+-----+--------+--------+---------+---------+--------+------+---------+-----------------+----------+--------+----------+----------+---------+--------+----+--------+----------+----------+----------+-----+---------+--------+---------+---------+-----+------+----------+-------+----------+----------+----------+---------+---------+---------+----------+---------+----------+--------+-------------------+----------+------------------------------+----------+---------------------+----------+---------------------------------------------------+---+----------+----------+----------+---------+---